# تجربة إثبات جدوى: توليد صورة «بعد» تلتزم بالتصميم

**السؤال الذي يجيب عنه هذا الدفتر:** هل يستطيع نموذج توليد مفتوح ومجاني أن يرسم أسناناً واقعية **داخل حدود التصميم الهندسي بالضبط**، دون أن يغيّر الشفاه أو الأسنان السفلية؟

**شرط النجاح** (على 20 صورة):
- الفحص الآلي: تطابق حدود الأسنان المولّدة مع التصميم (IoU) ≥ 0.80، وتغيّر خارج المنطقة المسموحة < 1، ولا أسنان زائدة، في **80% من الصور على الأقل**.
- الواقعية: تقييم أعمى من الطبيب (حزمة التقييم تُنشأ في آخر الدفتر).

**الخصوصية:** لا تستخدم صور مرضى في هذه التجربة. استخدم صورك أو صور أشخاص وافقوا، أو صوراً عامة مرخّصة.

**التشغيل:** Runtime ← Change runtime type ← **T4 GPU**، ثم شغّل الخلايا بالترتيب. المدة نحو 30–45 دقيقة.

In [ ]:
# 1) التجهيز
import subprocess, sys, os
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
!pip -q install mediapipe diffusers accelerate onnxruntime

In [ ]:
%%writefile /content/dsd_gen.py
"""Generation proof-of-concept: everything except the diffusion model itself.

Pipeline for one smile photo:
    face landmarks -> mouth crop (eye-line aligned) -> tooth segmentation (our U-Net, ONNX)
    -> geometric design (one polygon per tooth, in crop pixels) -> design validation
    -> prepaint + inpaint mask + edge map  (inputs for the diffusion model)
    -> [diffusion model in the notebook]
    -> composite back + automatic check of the generated teeth against the design.

The design is decided here, before any generation. The generator only renders it,
and check_generated() measures whether it kept to it.
"""
import math
import numpy as np
import cv2

TEETH = ['13', '12', '11', '21', '22', '23']          # model classes 1..6, image left -> right
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
SHADES = {'BL2': '#F3EFE6', 'B1': '#F0E9D8', 'A1': '#EEE6D4', 'A2': '#E9DDC4'}

# MediaPipe FaceLandmarker indices (478-point mesh)
IRIS_A, IRIS_B = 468, 473
COMM_A, COMM_B = 61, 291
SUBNASALE = 2
INNER_UPPER = [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
INNER_LOWER = [308, 324, 318, 402, 317, 14, 87, 178, 88, 95, 78]


# ---------------------------------------------------------------- face + crop
def load_rgb(path):
    """Read a photo with its EXIF rotation applied (phone photos are often stored sideways)."""
    from PIL import Image, ImageOps
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return np.asarray(im)


class FaceFinder:
    def __init__(self, task_path):
        import mediapipe as mp
        from mediapipe.tasks.python import BaseOptions, vision
        self.mp = mp
        self.det = vision.FaceLandmarker.create_from_options(
            vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=task_path), num_faces=1))

    def __call__(self, rgb):
        res = self.det.detect(self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
        if not res.face_landmarks:
            return None
        h, w = rgb.shape[:2]
        return np.array([[p.x * w, p.y * h] for p in res.face_landmarks[0]], np.float64)


def inner_lip_polygon(P):
    return np.concatenate([P[INNER_UPPER], P[INNER_LOWER[1:-1]]])


def mouth_crop(P, aspect=1.6, widen=1.18, out_w=768):
    """Affine crop around the mouth, rotated so the eye line is horizontal.
    Returns M (crop -> image) and Minv (image -> crop), both 2x3."""
    a, b = sorted([P[IRIS_A], P[IRIS_B]], key=lambda p: p[0])
    roll = math.atan2(b[1] - a[1], b[0] - a[0])
    c = inner_lip_polygon(P).mean(0)
    wc = np.linalg.norm(P[COMM_A] - P[COMM_B]) * widen
    out_h = int(round(out_w / aspect / 8) * 8)
    s = wc / out_w
    cs, sn = math.cos(roll) * s, math.sin(roll) * s
    M = np.array([[cs, -sn, c[0] - cs * out_w / 2 + sn * out_h / 2],
                  [sn, cs, c[1] - sn * out_w / 2 - cs * out_h / 2]])
    return M, cv2.invertAffineTransform(M), (out_w, out_h)


def warp_crop(rgb, M, size):
    return cv2.warpAffine(rgb, M, size, flags=cv2.INTER_CUBIC | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REFLECT)


def to_crop(Minv, pts):
    pts = np.asarray(pts, np.float64)
    return pts @ Minv[:, :2].T + Minv[:, 2]


def paste_back(full, crop_new, M, mask_crop):
    """Put the edited crop back into the full photo, only where mask_crop (0..1) says so."""
    h, w = full.shape[:2]
    Minv = cv2.invertAffineTransform(M)
    warped = cv2.warpAffine(crop_new, Minv, (w, h), flags=cv2.INTER_CUBIC | cv2.WARP_INVERSE_MAP)
    alpha = cv2.warpAffine(mask_crop.astype(np.float32), Minv, (w, h), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP)[..., None]
    return (full * (1 - alpha) + warped * alpha).astype(np.uint8)


# ---------------------------------------------------------------- segmentation
class Segmenter:
    """Our U-Net (MobileNetV2 encoder, 7 classes) exported to ONNX, input 1x3x384x384."""

    def __init__(self, onnx_path):
        import onnxruntime as ort
        self.s = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        self.name = self.s.get_inputs()[0].name

    def labels(self, crop_rgb):
        h, w = crop_rgb.shape[:2]
        x = cv2.resize(crop_rgb, (384, 384), interpolation=cv2.INTER_AREA).astype(np.float32) / 255
        x = ((x - MEAN) / STD).transpose(2, 0, 1)[None]
        lo = self.s.run(None, {self.name: x})[0][0]
        return cv2.resize(lo.argmax(0).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)

    def tooth_masks(self, crop_rgb, mouth=None, min_frac=0.002):
        lab = self.labels(crop_rgb)
        if mouth is not None:
            lab[mouth == 0] = 0
        out = {}
        for k, t in enumerate(TEETH, start=1):
            m = largest_component((lab == k).astype(np.uint8))
            if m.sum() >= min_frac * m.size:
                out[t] = m
        return out


def largest_component(m):
    n, cc, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.zeros_like(m)
    return (cc == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)


def box(m):
    ys, xs = np.nonzero(m)
    return dict(x0=float(xs.min()), x1=float(xs.max() + 1), y0=float(ys.min()), y1=float(ys.max() + 1))


def poly_mask(poly, w, h):
    m = np.zeros((h, w), np.uint8)
    cv2.fillPoly(m, [np.round(np.asarray(poly)).astype(np.int32)], 1)
    return m


# ---------------------------------------------------------------- geometric design
PROPORTION = {'red70': (1.0, 0.70, 0.49), 'golden': (1.0, 0.618, 0.382), 'natural': (1.0, 0.78, 0.62)}


def propose_design(teeth, mouth_poly, midline_x, shape='square', proportion='red70', wl=0.80, keep_span=True):
    """Target box for each of the six teeth, in crop pixels.

    teeth: current tooth boxes {'11': {x0,x1,y0,y1}, ...} (any subset; empty = no teeth visible)
    widths follow the chosen apparent-width proportion, centred on the facial midline;
    length from width/length ratio wl; incisal edges follow a smile arc (laterals shorter)."""
    p = PROPORTION[proportion]
    xs = mouth_poly[:, 0]
    inner_w = xs.max() - xs.min()
    cur_w = {t: b['x1'] - b['x0'] for t, b in teeth.items()}
    if keep_span and '13' in teeth and '23' in teeth:
        span = teeth['23']['x1'] - teeth['13']['x0']
        wc = span / (2 * sum(p))
    elif '11' in cur_w or '21' in cur_w:
        wc = np.mean([cur_w[t] for t in ('11', '21') if t in cur_w])
    else:                                           # no centrals: estimate from the mouth opening
        wc = 0.17 * inner_w / 0.85
    widths = {'11': wc * p[0], '21': wc * p[0], '12': wc * p[1], '22': wc * p[1], '13': wc * p[2], '23': wc * p[2]}
    L = wc / wl
    cen = [teeth[t]['y1'] for t in ('11', '21') if t in teeth]
    edge = float(np.mean(cen)) if cen else float(np.min(mouth_poly[:, 1]) + 0.9 * L)
    top_c = edge - L
    raise_ = {'11': 0, '21': 0, '12': 0.10 * L, '22': 0.10 * L, '13': 0.05 * L, '23': 0.05 * L}   # smile arc
    drop = {'11': 0, '21': 0, '12': 0.07 * L, '22': 0.07 * L, '13': 0, '23': 0}                  # gingival line
    out, x = {}, midline_x
    for t in ('21', '22', '23'):
        out[t] = dict(x0=x, x1=x + widths[t]); x += widths[t]
    x = midline_x
    for t in ('11', '12', '13'):
        out[t] = dict(x0=x - widths[t], x1=x); x -= widths[t]
    for t, b in out.items():
        b['y0'] = top_c + drop[t]
        b['y1'] = edge - raise_[t]
        b['shape'] = shape
    return out


def tooth_polygon(t, b, n=24):
    """Outline of one tooth inside its box: curved cervical line with the zenith slightly distal,
    rounded incisal corners (sharper mesially), canines with a cusp tip."""
    x0, x1, y0, y1, shape = b['x0'], b['x1'], b['y0'], b['y1'], b.get('shape', 'square')
    w, h = x1 - x0, y1 - y0
    right_side = t[0] == '2'                       # patient's left = image right
    mesial_is_left = right_side                     # mesial faces the midline
    r_dist = {'square': 0.18, 'oval': 0.40, 'triangular': 0.30}[shape] * w
    r_mes = {'square': 0.08, 'oval': 0.30, 'triangular': 0.15}[shape] * w
    taper = {'square': 0.06, 'oval': 0.10, 'triangular': 0.18}[shape] * w
    rl, rr = (r_mes, r_dist) if mesial_is_left else (r_dist, r_mes)
    pts = []
    # cervical arc, left to right, zenith shifted distally
    zx = x0 + w * (0.58 if mesial_is_left else 0.42)
    for i in range(n + 1):
        u = i / n
        xx = x0 + taper + (w - 2 * taper) * u
        k = (xx - zx) / (w / 2)
        pts.append((xx, y0 + 0.10 * h * min(1.0, k * k)))
    # right side down, incisal edge right to left, left side up
    pts.append((x1, y0 + 0.45 * h))
    for i in range(n // 2 + 1):
        a = math.pi / 2 * i / (n // 2)
        pts.append((x1 - rr + rr * math.cos(a), y1 - rr + rr * math.sin(a)))
    if t in ('13', '23'):                          # canine cusp
        pts.append((x0 + w / 2, y1 + 0.06 * h))
    for i in range(n // 2 + 1):
        a = math.pi / 2 + math.pi / 2 * i / (n // 2)
        pts.append((x0 + rl + rl * math.cos(a), y1 - rl + rl * math.sin(a)))
    pts.append((x0, y0 + 0.45 * h))
    return np.array(pts, np.float64)


def validate_design(design, mouth_poly, w, h):
    """Checks that run BEFORE any generation. Returns (ok, problems)."""
    problems = []
    mouth = poly_mask(mouth_poly, w, h)
    if len(design) != 6:
        problems.append(f'expected 6 teeth, got {len(design)}')
    xs = [(t, b['x0'], b['x1']) for t, b in design.items()]
    order = [t for t, _, _ in sorted(xs, key=lambda r: r[1])]
    if order != TEETH:
        problems.append('tooth order is not 13,12,11,21,22,23')
    for t, b in design.items():
        if b['x1'] - b['x0'] < 4 or b['y1'] - b['y0'] < 4:
            problems.append(f'{t}: degenerate size')
    c11 = (design['11']['x1'] - design['11']['x0']); c21 = (design['21']['x1'] - design['21']['x0'])
    if abs(c11 - c21) > 0.03 * max(c11, c21):
        problems.append('centrals differ in width by more than 3%')
    vis = {t: (poly_mask(tooth_polygon(t, b), w, h) & mouth).sum() for t, b in design.items()}
    for t in ('11', '21'):
        if vis.get(t, 0) < 50:
            problems.append(f'{t}: not visible inside the mouth opening')
    return (len(problems) == 0), problems


# ---------------------------------------------------------------- generator inputs
def hex_rgb(h):
    h = h.lstrip('#')
    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)], np.float32)


def prepare_inputs(crop, design, cur_masks, mouth_poly, shade='B1', dilate=5):
    """Prepaint (design teeth painted flat with simple shading), inpaint mask, and edge map.

    The mask covers the old upper teeth and the new design, inside the lip opening and above
    the incisal line, so the lips, gums outside the mask and the lower teeth stay original."""
    h, w = crop.shape[:2]
    mouth = poly_mask(mouth_poly, w, h)
    design_m = np.zeros((h, w), np.uint8)
    tooth_ms = {}
    for t, b in design.items():
        m = poly_mask(tooth_polygon(t, b), w, h)
        tooth_ms[t] = m
        design_m |= m
    old = np.zeros((h, w), np.uint8)
    for m in cur_masks.values():
        old |= m
    region = cv2.dilate(design_m | old, np.ones((2 * dilate + 1,) * 2, np.uint8))
    # incisal line per column: nothing below max(design edge, old edge) + 3 px  (keeps the lower teeth)
    cut = np.full(w, -1, np.int32)
    for m in [design_m, old]:
        ys, xs = np.nonzero(m)
        if len(xs):
            np.maximum.at(cut, xs, ys)
    cut = cv2.dilate(cut.reshape(1, -1).astype(np.float32), np.ones((1, 2 * dilate + 1), np.uint8)).ravel()
    yy = np.arange(h)[:, None]
    region &= (yy <= (cut[None, :] + 3)).astype(np.uint8)
    mask = region & mouth

    base = hex_rgb(SHADES[shade])
    # where old teeth were but the design has none: gum above the new cervical line, dark embrasure below it
    hole = ((mask == 1) & (design_m == 0)).astype(np.uint8)
    pre = cv2.inpaint(crop, cv2.dilate(hole, np.ones((3, 3), np.uint8)), 7, cv2.INPAINT_TELEA).astype(np.float32)
    gum_line = np.full(w, np.inf)
    for t, b in design.items():
        xa, xb = int(max(0, b['x0'] - 4)), int(min(w, b['x1'] + 4))
        gum_line[xa:xb] = np.minimum(gum_line[xa:xb], b['y0'] + 0.22 * (b['y1'] - b['y0']))
    dark = (hole == 1) & (yy > gum_line[None, :])
    pre[dark] = (45, 28, 28)
    detail = crop.astype(np.float32) - cv2.GaussianBlur(crop, (0, 0), 3).astype(np.float32)
    for t, m in tooth_ms.items():
        b = design[t]
        ys, xs = np.nonzero(m & mask)
        if not len(xs):
            continue
        u = (xs - b['x0']) / max(1, b['x1'] - b['x0'])               # 0..1 across
        v = (ys - b['y0']) / max(1, b['y1'] - b['y0'])               # 0..1 down
        shade_k = 0.86 + 0.10 * np.sin(np.pi * np.clip(u, 0, 1)) + 0.06 * np.clip(v, 0, 1) - 0.08 * np.clip(0.25 - v, 0, 1) * 4
        col = base[None, :] * shade_k[:, None]
        inc = np.clip((v - 0.85) / 0.15, 0, 1)[:, None]                # translucent incisal edge
        col = col * (1 - 0.25 * inc) + np.array([150, 160, 175], np.float32) * 0.25 * inc
        on_old = old[ys, xs][:, None].astype(np.float32)               # keep the patient's own enamel texture
        pre[ys, xs] = col + 0.6 * on_old * detail[ys, xs]
    edges = np.zeros((h, w), np.uint8)
    for t, b in design.items():
        cv2.polylines(edges, [np.round(tooth_polygon(t, b)).astype(np.int32)], True, 255, 2)
    edges &= (mouth * 255).astype(np.uint8)
    # thin dark contact lines between neighbouring teeth
    ink = cv2.dilate(edges, np.ones((2, 2), np.uint8)) > 0
    pre[ink & (mask == 1)] *= 0.55
    return dict(prepaint=np.clip(pre, 0, 255).astype(np.uint8), mask=mask, edges=edges,
                design_mask=design_m & mouth, tooth_masks={t: m & mouth for t, m in tooth_ms.items()})


def feather(mask, px=4):
    m = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=max(1, px // 2)).astype(np.float32)
    k = 2 * px + 1
    return cv2.GaussianBlur(m, (k, k), 0)


def composite(crop, generated, mask, px=4):
    a = feather(mask, px)[..., None]
    return (crop * (1 - a) + generated * a).astype(np.uint8), a[..., 0]


# ---------------------------------------------------------------- automatic check after generation
def check_generated(seg, before, after, inputs, mouth_poly):
    """Did the generated image keep to the design?  Returns a dict of metrics and a verdict."""
    h, w = after.shape[:2]
    mouth = poly_mask(mouth_poly, w, h)
    got = seg.tooth_masks(after, mouth)
    ious = {}
    for t, dm in inputs['tooth_masks'].items():
        if dm.sum() < 30:
            continue
        g = got.get(t, np.zeros_like(dm))
        ious[t] = float((g & dm).sum() / max(1, (g | dm).sum()))
    gen_all = np.zeros((h, w), np.uint8)
    for m in got.values():
        gen_all |= m
    dm_all = inputs['design_mask']
    union_iou = float((gen_all & dm_all).sum() / max(1, (gen_all | dm_all).sum()))
    outside = (inputs['mask'] == 0)
    drift = float(np.abs(after.astype(np.int16) - before.astype(np.int16))[outside].mean())
    extra = int(((gen_all == 1) & (cv2.dilate(dm_all, np.ones((9, 9), np.uint8)) == 0)).sum())
    r = dict(iou_mean=float(np.mean(list(ious.values()))) if ious else 0.0,
             iou_min=float(min(ious.values())) if ious else 0.0,
             iou_union=union_iou, teeth_found=len(got), outside_change=drift, extra_tooth_px=extra,
             iou=ious)
    r['pass'] = bool(r['iou_union'] >= 0.80 and r['outside_change'] < 1.0 and extra < 0.01 * dm_all.sum() + 50)
    return r


# ---------------------------------------------------------------- one photo, up to the generator
def analyse(rgb, face, seg, shape='square', proportion='red70', shade='B1', out_w=768):
    P = face(rgb)
    if P is None:
        return dict(ok=False, reason='no face found')
    M, Minv, size = mouth_crop(P, out_w=out_w)
    crop = warp_crop(rgb, M, size)
    mouth_poly = to_crop(Minv, inner_lip_polygon(P))
    mouth = poly_mask(mouth_poly, *size)
    if mouth.sum() < 0.02 * mouth.size:
        return dict(ok=False, reason='mouth closed or smile too narrow')
    eyes = to_crop(Minv, np.array([P[IRIS_A], P[IRIS_B]]))
    mid_x = float(0.5 * eyes[:, 0].mean() + 0.5 * to_crop(Minv, P[[SUBNASALE]])[0, 0])
    cur = seg.tooth_masks(crop, mouth)
    cur_boxes = {t: box(m) for t, m in cur.items()}
    design = propose_design(cur_boxes, mouth_poly, mid_x, shape=shape, proportion=proportion)
    ok, problems = validate_design(design, mouth_poly, *size)
    inputs = prepare_inputs(crop, design, cur, mouth_poly, shade=shade)
    return dict(ok=ok, problems=problems, P=P, M=M, crop=crop, mouth_poly=mouth_poly, midline_x=mid_x,
                current=cur_boxes, design=design, inputs=inputs)


def overlay(crop, design, current=None, mouth_poly=None):
    """Picture for the doctor: current teeth (red), design outlines (green), lip opening (blue)."""
    v = crop.copy()
    if mouth_poly is not None:
        cv2.polylines(v, [np.round(mouth_poly).astype(np.int32)], True, (60, 120, 255), 1)
    for b in (current or {}).values():
        cv2.rectangle(v, (int(b['x0']), int(b['y0'])), (int(b['x1']), int(b['y1'])), (230, 60, 60), 1)
    for t, b in design.items():
        cv2.polylines(v, [np.round(tooth_polygon(t, b)).astype(np.int32)], True, (40, 220, 90), 2)
    return v

In [ ]:
# 2) ملفات النماذج من مستودع المشروع (نموذج تحديد الأسنان + كاشف الوجه)
import os, urllib.request
REPO = 'https://raw.githubusercontent.com/alnsyryfwzy55-source/digital-smile-design-ai/main/docs/'
FILES = {'teeth_unet_v2.onnx': REPO + 'teeth_unet_v2.onnx',
         'face_landmarker.task': REPO + 'vendor/face_landmarker.task'}
for name, url in FILES.items():
    if not os.path.exists(name):
        try:
            urllib.request.urlretrieve(url, name)
        except Exception as e:
            print(f'تعذّر تنزيل {name}: {e}\nارفعه يدوياً من مجلد docs في المستودع إلى /content')
    print(name, os.path.getsize(name) if os.path.exists(name) else 'غير موجود')
# raw.githubusercontent لا يخدم ملفات Git LFS؛ إن كان حجم النموذج صغيراً جداً فهو مؤشر LFS
assert os.path.getsize('teeth_unet_v2.onnx') > 10_000_000, 'ملف النموذج غير صحيح'

## 3) الصور
ارفع ملفاً مضغوطاً `photos.zip` فيه 20 صورة ابتسامة أمامية تظهر فيها الأسنان العلوية (JPG أو PNG)،
أو ضع الصور في Google Drive وغيّر `PHOTO_DIR`.

In [ ]:
import glob, zipfile
PHOTO_DIR = '/content/photos'
os.makedirs(PHOTO_DIR, exist_ok=True)
if not glob.glob(PHOTO_DIR + '/*'):
    from google.colab import files
    up = files.upload()                       # اختر photos.zip
    for n in up:
        if n.endswith('.zip'):
            zipfile.ZipFile(n).extractall(PHOTO_DIR)
PHOTOS = sorted(p for p in glob.glob(PHOTO_DIR + '/**/*', recursive=True) if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(PHOTOS), 'صورة')

In [ ]:
# 4) التحليل والتصميم الهندسي (قبل أي توليد)
import numpy as np, cv2, json, matplotlib.pyplot as plt
import dsd_gen as G
face = G.FaceFinder('face_landmarker.task')
seg = G.Segmenter('teeth_unet_v2.onnx')
SHAPE, PROPORTION, SHADE = 'square', 'red70', 'B1'

cases, skipped = [], []
for p in PHOTOS:
    rgb = G.load_rgb(p)
    r = G.analyse(rgb, face, seg, shape=SHAPE, proportion=PROPORTION, shade=SHADE)
    if not r.get('ok'):
        skipped.append((os.path.basename(p), r.get('reason') or r.get('problems')))
        continue
    r['path'], r['rgb'] = p, rgb
    cases.append(r)
print('جاهزة للتوليد:', len(cases), '| مستبعدة:', len(skipped))
for s in skipped: print('  ', s)

n = min(6, len(cases))
fig, ax = plt.subplots(n, 2, figsize=(12, 3.2 * n))
for i in range(n):
    c = cases[i]
    ax[i, 0].imshow(G.overlay(c['crop'], c['design'], c['current'], c['mouth_poly'])); ax[i, 0].set_title('التصميم (أخضر) فوق الأسنان الحالية (أحمر)')
    ax[i, 1].imshow(c['inputs']['prepaint']); ax[i, 1].set_title('مدخل التوليد')
    for a in ax[i]: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 5) نموذج التوليد: Stable Diffusion inpainting + ControlNet (حواف التصميم)
import torch
from PIL import Image
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, UniPCMultistepScheduler

BASES = {'A_inpaint': 'stable-diffusion-v1-5/stable-diffusion-inpainting',
         'B_base':    'stable-diffusion-v1-5/stable-diffusion-v1-5'}
controlnet = ControlNetModel.from_pretrained('lllyasviel/control_v11p_sd15_canny', torch_dtype=torch.float16)
_pipes = {}
def get_pipe(key):
    if key not in _pipes:
        for k in list(_pipes):                 # one pipeline in GPU memory at a time
            del _pipes[k]
        torch.cuda.empty_cache()
        p = StableDiffusionControlNetInpaintPipeline.from_pretrained(
            BASES[key], controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
        p.scheduler = UniPCMultistepScheduler.from_config(p.scheduler.config)
        p.set_progress_bar_config(disable=True)
        _pipes[key] = p.to('cuda')
    return _pipes[key]

PROMPT = ('close-up photo of a natural smile, healthy natural teeth, realistic enamel texture, '
          'subtle translucency at the edges, natural lighting, sharp focus, high detail')
NEGATIVE = ('cartoon, painting, 3d render, deformed teeth, extra teeth, fused teeth, missing teeth, gap, braces, '
            'metal, blurry, lowres, lipstick, distorted lips, tongue')

def match_color(gen, ref, mask, k=0.6):
    """Pull the generated teeth's mean colour towards the chosen shade (LAB), keeping texture."""
    g = cv2.cvtColor(gen, cv2.COLOR_RGB2LAB).astype(np.float32); r = cv2.cvtColor(ref, cv2.COLOR_RGB2LAB).astype(np.float32)
    m = mask.astype(bool)
    if m.sum() < 50: return gen
    g[m] += k * (r[m].mean(0) - g[m].mean(0))
    return cv2.cvtColor(np.clip(g, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

def render(case, base='A_inpaint', strength=0.6, cn_scale=0.8, seed=0, steps=30):
    inp, crop = case['inputs'], case['crop']
    h, w = crop.shape[:2]
    pipe = get_pipe(base)
    out = pipe(PROMPT, negative_prompt=NEGATIVE,
               image=Image.fromarray(inp['prepaint']), mask_image=Image.fromarray(inp['mask'] * 255),
               control_image=Image.fromarray(np.dstack([inp['edges']] * 3)),
               height=h, width=w, strength=strength, num_inference_steps=steps, guidance_scale=7.0,
               controlnet_conditioning_scale=cn_scale, generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    gen = np.asarray(out.resize((w, h)))
    gen = match_color(gen, inp['prepaint'], inp['design_mask'])
    after, alpha = G.composite(crop, gen, inp['mask'])
    chk = G.check_generated(seg, crop, after, inp, case['mouth_poly'])
    return after, alpha, chk

In [ ]:
# 6) مسح الإعدادات على أول 5 صور: أي إعداد يلتزم بالتصميم أكثر؟
import itertools, time, pandas as pd
SWEEP = list(itertools.product(['A_inpaint', 'B_base'], [0.45, 0.6, 0.75], [0, 1]))
rows = []
t0 = time.time()
for base, s, seed in SWEEP:
    for i, c in enumerate(cases[:5]):
        _, _, chk = render(c, base=base, strength=s, seed=seed)
        rows.append(dict(base=base, strength=s, seed=seed, case=i, **{k: v for k, v in chk.items() if k != 'iou'}))
df = pd.DataFrame(rows)
summary = df.groupby(['base', 'strength']).agg(pass_rate=('pass', 'mean'), iou_union=('iou_union', 'mean'),
                                               iou_min=('iou_min', 'mean'), outside=('outside_change', 'mean')).reset_index()
summary = summary.sort_values(['pass_rate', 'iou_union'], ascending=False)
print(f'{time.time() - t0:.0f} ث'); display(summary)
BEST = summary.iloc[0][['base', 'strength']].to_dict()
print('أفضل إعداد:', BEST)

In [ ]:
# 7) الإعداد الأفضل على كل الصور: 3 بذور لكل صورة، ويُختار أعلاها التزاماً بالتصميم
OUT = '/content/results'; os.makedirs(OUT, exist_ok=True)
results = []
for i, c in enumerate(cases):
    best = None
    for seed in range(3):
        after, alpha, chk = render(c, base=BEST['base'], strength=float(BEST['strength']), seed=seed)
        if best is None or chk['iou_union'] > best[2]['iou_union']:
            best = (after, alpha, chk, seed)
    after, alpha, chk, seed = best
    name = f'case_{i:02d}'
    full_after = G.paste_back(c['rgb'], after, c['M'], alpha)
    sheet = np.concatenate([c['crop'], G.overlay(c['crop'], c['design']), after], 1)
    cv2.imwrite(f'{OUT}/{name}_sheet.jpg', cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'{OUT}/{name}_after_full.jpg', cv2.cvtColor(full_after, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
    results.append(dict(case=name, photo=os.path.basename(c['path']), seed=seed, **{k: v for k, v in chk.items() if k != 'iou'},
                        iou_per_tooth=chk['iou']))
res = pd.DataFrame(results)
res.drop(columns=['iou_per_tooth']).to_csv(f'{OUT}/metrics.csv', index=False)
print(f"اجتاز الفحص الآلي: {res['pass'].sum()} من {len(res)} ({100 * res['pass'].mean():.0f}%)  |  الشرط: 80%")
display(res[['case', 'iou_union', 'iou_min', 'teeth_found', 'outside_change', 'extra_tooth_px', 'pass']])

In [ ]:
# 8) عرض النتائج: قبل | التصميم | بعد
for r in results[:8]:
    im = cv2.cvtColor(cv2.imread(f"{OUT}/{r['case']}_sheet.jpg"), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 3.6)); plt.imshow(im); plt.axis('off')
    plt.title(f"{r['case']}  IoU={r['iou_union']:.2f}  {'✓' if r['pass'] else '✗'}"); plt.show()

In [ ]:
# 9) حزمة التقييم الأعمى للطبيب: صور حقيقية ومولّدة مخلوطة بأسماء عشوائية
import random, shutil, csv
BLIND = f'{OUT}/blind_test'; os.makedirs(BLIND, exist_ok=True)
items = []
for i, c in enumerate(cases):
    after = cv2.imread(f'{OUT}/case_{i:02d}_sheet.jpg')
    w3 = after.shape[1] // 3
    items.append(('real', i, cv2.cvtColor(c['crop'], cv2.COLOR_RGB2BGR)))
    items.append(('generated', i, after[:, 2 * w3:]))
random.seed(7); random.shuffle(items)
key = []
for k, (kind, i, im) in enumerate(items):
    fn = f'img_{k:03d}.jpg'; cv2.imwrite(f'{BLIND}/{fn}', im); key.append((fn, kind, i))
with open(f'{OUT}/blind_key_DO_NOT_SHARE.csv', 'w', newline='') as f: csv.writer(f).writerows([('file', 'kind', 'case')] + key)
with open(f'{BLIND}/rating_sheet.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['file', 'realism_1_to_5', 'looks_generated_yes_no', 'notes'])
    for fn, _, _ in key: w.writerow([fn, '', '', ''])
print(len(key), 'صورة في', BLIND, '— أرسل المجلد للطبيب دون ملف المفتاح')

In [ ]:
# 10) تنزيل النتائج + ملخص ترسله لي
summary_out = dict(best_config=BEST, n=len(res), pass_rate=float(res['pass'].mean()),
                   iou_union_mean=float(res['iou_union'].mean()), iou_min_mean=float(res['iou_min'].mean()),
                   outside_change_max=float(res['outside_change'].max()), skipped=len(skipped),
                   sweep=summary.round(3).to_dict(orient='records'))
json.dump(summary_out, open(f'{OUT}/summary.json', 'w'), indent=1)
print(json.dumps(summary_out, indent=1))
shutil.make_archive('/content/generation_results', 'zip', OUT)
from google.colab import files; files.download('/content/generation_results.zip')